In [ ]:
import subprocess, shlex, os, glob, pathlib, textwrap

R_SNIPPET = """
args <- commandArgs(trailingOnly=TRUE)
in_path  <- args[1]
out_path <- args[2]
df <- readRDS(in_path)

# write to CSV instead of parquet
library(readr)
write_csv(df, out_path)
"""

def rds_to_csv_via_rscript(in_path, out_path):
    script = textwrap.dedent(R_SNIPPET)
    cmd = f'Rscript -e {shlex.quote(script)} {shlex.quote(in_path)} {shlex.quote(out_path)}'
    subprocess.check_call(cmd, shell=True)
    print(f"✅ {in_path} → {out_path}")

DATA_DIR = "data"
for in_path in glob.glob(os.path.join(DATA_DIR, "*.rds")):
    out_path = os.path.splitext(in_path)[0] + ".csv"
    pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    rds_to_csv_via_rscript(in_path, out_path)


✅ data/fulldata_adm1_africa.rds → data/fulldata_adm1_africa.csv
✅ data/fulldata_global.rds → data/fulldata_global.csv
✅ data/fulldata_adm0_africa.rds → data/fulldata_adm0_africa.csv


In [ ]:
import pandas as pd
import os, glob, pathlib

DATA_DIR = "data"

def csv_to_parquet(in_path, out_path):
    # Read CSV
    df = pd.read_csv(in_path)
    # Write Parquet
    df.to_parquet(out_path, index=False)
    print(f"✅ {in_path} → {out_path}")

for in_path in glob.glob(os.path.join(DATA_DIR, "*.csv")):
    out_path = os.path.splitext(in_path)[0] + ".parquet"
    pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    csv_to_parquet(in_path, out_path)

✅ data/fulldata_global.csv → data/fulldata_global.parquet
✅ data/fulldata_adm1_africa.csv → data/fulldata_adm1_africa.parquet
✅ data/fulldata_adm0_africa.csv → data/fulldata_adm0_africa.parquet


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ----------------------------
# 1) Configure where your data lives
# ----------------------------
DATA_PATH = "data/fulldata_adm0_africa.parquet"  # change to your file path

# ----------------------------
# 2) Helpers to load the original full data in Python
# ----------------------------
df = pd.read_parquet(DATA_PATH)

# Ensure expected columns exist
required_base = ["country_name","gwno","year","month","yearmonth","sri_num"]
missing_base = [c for c in required_base if c not in df.columns]
if missing_base:
    raise SystemExit(f"[!] Missing required columns: {missing_base}")

# ----------------------------
# 3) Define variable groups exactly like in your R code
# ----------------------------
# Outcomes (you can switch to sbv_fat_be, osv_fat_be, nsv_fat_be if desired)
TARGET = "sri_num"

# Benchmark (BM) predictors (lagged outcomes)
bm_vars = [
    "sbv_fat_be_lag", "osv_fat_be_lag", "nsv_fat_be_lag", "sri_num_lag", "sri_fat_lag"
]

# Covariate (COV) predictors (full set from your non-log script)
cov_vars = [
    "cinc", "elev_mean", "ethfrac", "ethpol", "farmland", "forest", "irregular",
    "irst", "milex", "milper", "n_leaders", "nbuiltup", "nethgr", "newlmtnest", "npetro",
    "open_terrain", "pec", "relfrac", "relpol", "road_density", "road_length", "rugged",
    "sum_igo_anytype", "sum_igo_associate", "sum_igo_full", "sum_igo_observer",
    "tpop", "upop", "v2x_polyarchy", "wbgdp2011est", "wbgdppc2011est", "wbpopest", "xm_qudsest",
    "l1_irregular", "l1_leadertransition", "l1_n_leaders", "l1_v2x_polyarchy",
    "l1_wbgdppc2011est", "l1_wbpopest", "l1_xm_qudsest"
]

# Google Trends + Wikipedia (GTW) predictors
gtw_vars = [c for c in df.columns if (c.startswith("views") or c.startswith("hits")) and not (c.endswith("log") or c.endswith("change"))]

# Filter to keep only existing columns (robust to minor name differences)
bm_vars = [c for c in bm_vars if c in df.columns]
cov_vars = [c for c in cov_vars if c in df.columns]
gtw_vars = [c for c in gtw_vars if c in df.columns]

# Safety check: drop rows with any NA in the modeling columns we’ll use in each run
id_cols = ["country_name","gwno","year","month","yearmonth"]

# ----------------------------
# 4) Metric functions (RMSE, MAE, AC, CCC, RAC)
# ----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

# Lin's concordance correlation coefficient (CCC)
def ccc(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    mu_x = y_true.mean()
    mu_y = y_pred.mean()
    s_x2 = y_true.var(ddof=1)
    s_y2 = y_pred.var(ddof=1)
    s_xy = np.cov(y_true, y_pred, ddof=1)[0,1]
    return (2 * s_xy) / (s_x2 + s_y2 + (mu_x - mu_y)**2 + 1e-12)

# Accuracy coefficient (AC) = Lin's accuracy component C_b (so CCC = r * AC)
def accuracy_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    s_x = y_true.std(ddof=1)
    s_y = y_pred.std(ddof=1)
    mu_x = y_true.mean()
    mu_y = y_pred.mean()
    if s_x == 0 or s_y == 0:
        return 0.0
    v = (s_y / s_x) + (s_x / s_y) + ((mu_y - mu_x)**2) / (s_x * s_y)
    return 2.0 / (v + 1e-12)

# Robinson’s Agreement Coefficient (RAC).
# Here we use the widely adopted "Willmott-style" agreement denominator,
# which has been referred to as a Robinson/Agreement coefficient in forecasting work:
# RAC = 1 - sum((y - ŷ)^2) / sum((|ŷ - ȳ| + |y - ȳ|)^2)
def rac(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    ybar = y_true.mean()
    num = np.sum((y_true - y_pred)**2)
    den = np.sum((np.abs(y_pred - ybar) + np.abs(y_true - ybar))**2) + 1e-12
    return 1 - num / den

# ----------------------------
# 5) Modeling grids akin to your mtry sequences (clamped to the feature count)
# ----------------------------
def mtry_grid(n_features, base_seq):
    # clamp each candidate to [1, n_features]
    uniq = sorted({max(1, min(n_features, int(x))) for x in base_seq})
    # also include "sqrt" and "log2" styled choices via fractions if appropriate
    if n_features >= 3:
        uniq = sorted(set(uniq + [int(np.sqrt(n_features)), max(1,int(np.log2(n_features)))]) )
    return uniq

# Sequences ported from your R code
seqs = {
    "bm":       list(range(1,6)),                  # 1..5
    "cov":      list(range(5,41,5)),               # 5..40 step 5
    "gtw":      list(range(5,21,5)),               # 5..20
    "bm_gtw":   list(range(5,26,5)),               # 5..25
    "bm_cov":   list(range(5,46,5)),               # 5..45
    "cov_gtw":  list(range(5,61,5)),               # 5..60
    "bm_cov_gtw": list(range(5,66,5)),             # 5..65
}

def build_feature_sets():
    combos = {
        "bm": bm_vars,
        "cov": cov_vars,
        "gtw": gtw_vars,
        "bm_gtw": bm_vars + gtw_vars,
        "cov_gtw": cov_vars + gtw_vars,
        "bm_cov": bm_vars + cov_vars,
        "bm_cov_gtw": bm_vars + cov_vars + gtw_vars
    }
    # drop duplicates while preserving order
    for k,v in combos.items():
        seen, dedup = set(), []
        for col in v:
            if col not in seen:
                dedup.append(col); seen.add(col)
        combos[k] = dedup
    return combos

feature_sets = build_feature_sets()

# ----------------------------
# 6) Train/Test split + model selection + metrics for each year & model
# ----------------------------
YEARS = [2020, 2021, 2022, 2023]

results = []
predictions_by_run = {}  # optional: to let you inspect per-run preds later

for heldout_year in YEARS:
    # Train: all <= heldout_year-1; Test: == heldout_year
    train_idx = df["year"] <= (heldout_year - 1)
    test_idx  = df["year"] == heldout_year

    df_train = df.loc[train_idx].copy()
    df_test  = df.loc[test_idx].copy()

    if df_test.empty or df_train.empty:
        print(f"[!] Skipping {heldout_year}: train or test is empty.")
        continue

    for model_name, cols in feature_sets.items():
        if len(cols) == 0:
            print(f"[!] Skipping {model_name}: no matching feature columns in data.")
            continue

        # drop rows with any NA in target or features (no intermediate datasets saved)
        needed = cols + [TARGET]
        df_train_clean = df_train.dropna(subset=needed)
        df_test_clean  = df_test.dropna(subset=needed)

        if df_test_clean.empty or df_train_clean.empty:
            print(f"[!] {heldout_year} / {model_name}: no rows after dropping NAs, skipping.")
            continue

        X_train = df_train_clean[cols].values
        y_train = df_train_clean[TARGET].values
        X_test  = df_test_clean[cols].values
        y_test  = df_test_clean[TARGET].values

        # Build mtry (max_features) grid per model, clamped to feature count
        nfeat = X_train.shape[1]
        grid_mtry = mtry_grid(nfeat, seqs[model_name if model_name in seqs else "bm"])

        # Define RF; tune only max_features & min_samples_leaf (analogous to min.node.size)
        param_grid = {
            "max_features": grid_mtry,
            "min_samples_leaf": [5, 10],   # mirrors your 5 & 10
            "max_depth": [None],           # keep trees deep; RF handles overfit via averaging
        }

        base = RandomForestRegressor(
            n_estimators=1000,
            n_jobs=-1,
            random_state=815,
            bootstrap=True
        )

        # 3-fold CV on the training set (shuffled=False to respect time-ish ordering)
        cv = KFold(n_splits=3, shuffle=False)
        gs = GridSearchCV(base, param_grid, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1, verbose=1)
        gs.fit(X_train, y_train)

        best_model = gs.best_estimator_
        y_pred = best_model.predict(X_test)

        # Metrics
        metric_rmse = rmse(y_test, y_pred)
        metric_mae  = mae(y_test, y_pred)
        metric_ccc  = ccc(y_test, y_pred)
        metric_ac   = accuracy_coefficient(y_test, y_pred)
        metric_rac  = rac(y_test, y_pred)

        results.append({
            "heldout_year": heldout_year,
            "model": model_name,
            "n_features": nfeat,
            "best_max_features": best_model.get_params().get("max_features"),
            "best_min_samples_leaf": best_model.get_params().get("min_samples_leaf"),
            "RMSE": metric_rmse,
            "MAE": metric_mae,
            "AC": metric_ac,
            "CCC": metric_ccc,
            "RAC": metric_rac
        })

        # stash predictions (optional)
        key = (heldout_year, model_name)
        predictions_by_run[key] = pd.DataFrame({
            "country_name": df_test_clean["country_name"].values if "country_name" in df_test_clean.columns else np.nan,
            "gwno": df_test_clean["gwno"].values if "gwno" in df_test_clean.columns else np.nan,
            "year": df_test_clean["year"].values,
            "month": df_test_clean["month"].values if "month" in df_test_clean.columns else np.nan,
            "yearmonth": df_test_clean["yearmonth"].values if "yearmonth" in df_test_clean.columns else np.nan,
            "y_true": y_test,
            "y_pred": y_pred
        })

# Aggregate results into a tidy table
res_df = pd.DataFrame(results)

# Order columns nicely
cols_order = ["heldout_year","model","n_features","best_max_features","best_min_samples_leaf",
                "RMSE","MAE","AC","CCC","RAC"]
res_df = res_df[cols_order].sort_values(["heldout_year","model"]).reset_index(drop=True)

# Show a compact summary (rounded)
display_df = res_df.copy()
for m in ["RMSE","MAE","AC","CCC","RAC"]:
    display_df[m] = display_df[m].astype(float).round(4)

# Print the first few rows here for quick glance
display_df

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Fitting 3 folds for each of 10 candidates, totalling 30 fits
